In [0]:
%run "../Modulo 2/Configurando o storage"

In [0]:
caminho_bronze = "abfss://bronze@datalakexperiun.dfs.core.windows.net/"
caminho_silver = "abfss://silver@datalakexperiun.dfs.core.windows.net/"
caminho_gold = "abfss://gold@datalakexperiun.dfs.core.windows.net/"
caminho_arquivo_times = f"{caminho_bronze}times.json"


In [0]:
from pyspark.sql.types import StructField, StructType, StringType, DoubleType, BooleanType

In [0]:
schema_times = StructType(
    fields=[
        StructField("id", StringType(), False),
        StructField("name", StringType(), False),
        StructField("code", StringType(), False),
        StructField("country", StringType(), False),
        StructField("founded", DoubleType(), False),
        StructField("national", BooleanType(), False),
        StructField("logo", StringType(), False),
    ]
)

In [0]:
times_df = spark.read.schema(schema_times).json(caminho_arquivo_times)

In [0]:
display(times_df)

In [0]:
times_df.printSchema()

In [0]:
times_df = times_df.withColumn('founded', times_df.founded.cast('integer'))

In [0]:
times_df_renomeado = times_df \
    .withColumnRenamed("id", "id_time") \
    .withColumnRenamed("name", "nm_time") \
    .withColumnRenamed("code", "cd_time") \
    .withColumnRenamed("country", "nm_pais") \
    .withColumnRenamed("founded", "ano_fundacao") \
    .withColumnRenamed("national", "e_nacional") \
    .withColumnRenamed("logo", "link_logo")

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
times_df_data = times_df_renomeado.withColumn('dt_ingestao', current_timestamp())

In [0]:
display(times_df_data)

In [0]:
times_df_data.write.parquet(f"{caminho_silver}times/", mode="overwrite")